<a href="https://colab.research.google.com/github/javmencia/ReSTORELab/blob/main/SLElstmFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                            balanced_accuracy_score, cohen_kappa_score,
                            precision_score, recall_score, accuracy_score,
                            f1_score)
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking, BatchNormalization, Attention
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.metrics import Precision, Recall
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import tensorflow as tf
import random

# Set seeds for reproducibility
random.seed(1927)
np.random.seed(1927)
tf.random.set_seed(1927)

# Enhanced focal loss with class weighting
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fn(y_true, y_pred):
        epsilon = 1e-7
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Calculate cross entropy
        cross_entropy = -y_true * tf.math.log(y_pred)

        # Calculate weights
        weights = alpha * tf.pow(1. - y_pred, gamma)

        return tf.reduce_sum(weights * cross_entropy, axis=-1)
    return focal_loss_fn

# --- Data Loading & Enhanced Preprocessing ---
data = pd.read_csv('sledatacut2.csv', parse_dates=['ASSDT'], low_memory=False)
data = data.sort_values(['PTNO', 'ASSDT'])

# Enhanced time-based features
data['time_since_first'] = data.groupby('PTNO')['ASSDT'].transform(
    lambda x: (x - x.min()).dt.days
)
data['time_since_last'] = data.groupby('PTNO')['ASSDT'].transform(
    lambda x: x.diff().dt.days.fillna(0)
)


label_encoder = LabelEncoder()
data['end_state_encoded'] = label_encoder.fit_transform(data['end_state'])


# Enhanced steroid categories encoding
steroid_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
data['STEROID_CAT_numeric'] = data['STEROID_CAT'].map(steroid_mapping)

# Enhanced feature set with more domain-specific features
features = [
    'male', 'severe_flare', 'mild_flare', 'total_flares',
    'STERDOSE', 'INCEPT', 'AMDOSE',
    'age_at_record', 'time_since_last', 'time_since_first',
    'visit_num', 'score_n', 'AMS', "SLEDAI_Constitutional",
    "SLEDAI_Cutaneous", "SLEDAI_Musculoskeletal", "SLEDAI_Serositis",
    "SLEDAI_Renal", "SLEDAI_Neurological", "SLEDAI_Hematological",
    "SLEDAI_Immunological", "SLEDAI_Vascular", "SLEDAI_Other",
    'STEROID_CAT_numeric', 'BIOG_RIT', 'BIOG_BEL', 'BIOG_ABA', 'BIOG_OTH', 'am_chloroquine', 'am_hydroxychloroquine', 'am_atabrine',
    'STPNO', 'STPDOSE'
]

# Enhanced interaction terms
data['sdi_flare_interaction'] = data['score_n'] * data['total_flares']
data['steroid_flare_interaction'] = data['STEROID_CAT_numeric'] * data['total_flares']
data['age_flare_interaction'] = data['age_at_record'] * data['total_flares']

# Enhanced temporal features
data['flares_per_month'] = data['total_flares'] / (data['time_since_first']/30 + 1)
data['sdi_change_rate'] = data.groupby('PTNO', group_keys=False)['score_n'].apply(
    lambda x: x.diff().fillna(0)) / (data.groupby('PTNO')['time_since_last'].transform(lambda x: x/30 + 1e-6))

data['delta_SLEDAI_Renal'] = data.groupby('PTNO')['SLEDAI_Renal'].diff()

# Fill the first visit's delta with 0 (no previous visit to compare)
data['delta_SLEDAI_Renal'] = data['delta_SLEDAI_Renal'].fillna(0)

# Calculate time between consecutive visits in days
data['time_between_visits'] = data.groupby('PTNO')['ASSDT'].diff().dt.days

# Compute visits per year (frequency) for each patient
data['visits_per_year'] = data.groupby('PTNO')['ASSDT'].transform(
    lambda x: len(x) / ((x.max() - x.min()).days / 365.25 + 1e-6)  # Avoid division by zero
)

# Update features list
features += [
    'sdi_flare_interaction', 'steroid_flare_interaction', 'age_flare_interaction',
    'sdi_change_rate', 'delta_SLEDAI_Renal', 'visits_per_year'
]

# Handle missing values more robustly
for col in features:
    if data[col].isnull().any():
        if col in ['score_n', 'total_flares', 'severe_flare', 'mild_flare']:
            data[col] = data[col].fillna(0)
        else:
            data[col] = data.groupby('end_state_encoded')[col].transform(
                lambda x: x.fillna(x.median())
            )

# Enhanced scaling - using StandardScaler for features with large ranges
data[features] = data[features].fillna(0)

# Normalize features
scaler = MinMaxScaler()
data[features] = scaler.fit_transform(data[features])

# Enhanced sequence creation with padding and masking
def create_sequences(data, features, max_seq_length=None):
    patients = data['PTNO'].unique()
    num_features = len(features)

    if not max_seq_length:
        max_seq_length = data.groupby('PTNO').size().max()

    X = np.zeros((len(patients), max_seq_length, num_features))
    y = np.zeros(len(patients))
    seq_lengths = np.zeros(len(patients))

    for i, patient in enumerate(patients):
        patient_data = data[data['PTNO'] == patient].sort_values('ASSDT')
        seq_len = len(patient_data)
        seq_lengths[i] = seq_len

        # Fill the sequence data
        X[i, :seq_len, :] = patient_data[features].values

        # Get the end_state label (using first observation)
        y[i] = patient_data['end_state_encoded'].iloc[0]

    return X, to_categorical(y), seq_lengths

# Create sequences
X, y, seq_lengths = create_sequences(data, features)
y_labels = np.argmax(y, axis=1)

# --- Enhanced Cross-Validation Setup ---
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_results = {
    'lstm_val_accuracy': [],
    'lstm_val_balanced_accuracy': [],
    'lstm_val_f1_per_class': [],
    'lstm_val_precision_per_class': [],
    'lstm_val_recall_per_class': [],
    'lstm_models': [],

    'rf_val_accuracy': [],
    'rf_val_balanced_accuracy': [],
    'rf_val_f1_per_class': [],
    'rf_val_precision_per_class': [],
    'rf_val_recall_per_class': [],
    'rf_models': [],

    'ensemble_val_accuracy': [],
    'ensemble_val_balanced_accuracy': [],
    'ensemble_val_f1_per_class': []
}

# --- Enhanced Cross-Validation Loop ---
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_labels)):
    print(f"\n=== Fold {fold + 1}/{n_splits} ===")

    # Data splitting
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    seq_train, seq_val = seq_lengths[train_idx], seq_lengths[val_idx]

    y_train_labels = np.argmax(y_train, axis=1)
    y_val_labels = np.argmax(y_val, axis=1)

    # Enhanced class weighting
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)
    class_weight_dict = dict(enumerate(class_weights))

    # --- Enhanced LSTM Model ---
    model = Sequential([
        Masking(mask_value=0., input_shape=(None, len(features))),
        LSTM(128, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01)),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(64, return_sequences=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01)),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(32, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01)),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
        Dense(y.shape[1], activation='softmax')
    ])

    # Enhanced optimizer with gradient clipping
    optimizer = Adam(learning_rate=0.001, clipvalue=0.5)

    model.compile(
        optimizer=optimizer,
        loss=focal_loss(gamma=2., alpha=0.5),
        metrics=['accuracy',
                Precision(name='precision'),
                Recall(name='recall')]
    )

    # Enhanced callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f'best_model_fold{fold}.h5', save_best_only=True, monitor='val_accuracy')
    ]

    # Enhanced training with class weights
    history = model.fit(
        X_train, y_train,
        batch_size=32,
        epochs=200,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )

    # LSTM Evaluation
    y_val_pred_lstm = model.predict(X_val, verbose=0)
    y_val_pred_lstm_classes = np.argmax(y_val_pred_lstm, axis=1)

    # Store LSTM metrics
    cv_results['lstm_val_accuracy'].append(accuracy_score(y_val_labels, y_val_pred_lstm_classes))
    cv_results['lstm_val_balanced_accuracy'].append(balanced_accuracy_score(y_val_labels, y_val_pred_lstm_classes))
    cv_results['lstm_val_precision_per_class'].append(precision_score(y_val_labels, y_val_pred_lstm_classes, average=None, zero_division=0))
    cv_results['lstm_val_recall_per_class'].append(recall_score(y_val_labels, y_val_pred_lstm_classes, average=None, zero_division=0))
    cv_results['lstm_val_f1_per_class'].append(f1_score(y_val_labels, y_val_pred_lstm_classes, average=None, zero_division=0))
    cv_results['lstm_models'].append(model)

    # --- Enhanced Random Forest with SMOTE ---
    # --- Enhanced Random Forest with SMOTE ---
    # Get mean of all observations for each patient for RF
    X_train_mean = np.array([np.mean(X_train[i, :int(seq_train[i]), :], axis=0) for i in range(X_train.shape[0])])
    X_val_mean = np.array([np.mean(X_val[i, :int(seq_val[i]), :], axis=0) for i in range(X_val.shape[0])])

    # Create pipeline with SMOTE and Balanced RF
    rf_pipeline = Pipeline([
        ('smote', SMOTE(sampling_strategy='auto', random_state=42)),
        ('rf', BalancedRandomForestClassifier(
            n_estimators=500,
            sampling_strategy='all',
            replacement=True,
            random_state=42,
            class_weight='balanced_subsample',
            max_depth=15,
            min_samples_leaf=3,
            max_features='sqrt'
        ))
    ])

    rf_pipeline.fit(X_train_mean, y_train_labels)

    # RF Evaluation
    y_val_pred_rf = rf_pipeline.predict(X_val_mean)
    y_val_pred_rf_proba = rf_pipeline.predict_proba(X_val_mean)

    # Store RF metrics
    cv_results['rf_val_accuracy'].append(accuracy_score(y_val_labels, y_val_pred_rf))
    cv_results['rf_val_balanced_accuracy'].append(balanced_accuracy_score(y_val_labels, y_val_pred_rf))
    cv_results['rf_val_precision_per_class'].append(precision_score(y_val_labels, y_val_pred_rf, average=None, zero_division=0))
    cv_results['rf_val_recall_per_class'].append(recall_score(y_val_labels, y_val_pred_rf, average=None, zero_division=0))
    cv_results['rf_val_f1_per_class'].append(f1_score(y_val_labels, y_val_pred_rf, average=None, zero_division=0))
    cv_results['rf_models'].append(rf_pipeline)


    # --- Ensemble Evaluation ---
    ensemble_proba = 0.8 * y_val_pred_lstm + 0.2 * y_val_pred_rf_proba
    y_val_pred_ensemble = np.argmax(ensemble_proba, axis=1)

    cv_results['ensemble_val_accuracy'].append(accuracy_score(y_val_labels, y_val_pred_ensemble))
    cv_results['ensemble_val_balanced_accuracy'].append(balanced_accuracy_score(y_val_labels, y_val_pred_ensemble))
    cv_results['ensemble_val_f1_per_class'].append(f1_score(y_val_labels, y_val_pred_ensemble, average=None, zero_division=0))

    # --- Fold Summary ---
    print(f"LSTM Val Accuracy: {cv_results['lstm_val_accuracy'][-1]:.4f}")
    print(f"RF Val Accuracy: {cv_results['rf_val_accuracy'][-1]:.4f}")
    print(f"Ensemble Val Accuracy: {cv_results['ensemble_val_accuracy'][-1]:.4f}")
    print(f"LSTM Val Balanced Accuracy: {cv_results['lstm_val_balanced_accuracy'][-1]:.4f}")
    print(f"RF Val Balanced Accuracy: {cv_results['rf_val_balanced_accuracy'][-1]:.4f}")
    print(f"Ensemble Val Balanced Accuracy: {cv_results['ensemble_val_balanced_accuracy'][-1]:.4f}")



# --- Final Enhanced Evaluation ---
print("\n=== Final Cross-Validation Results ===")

# LSTM Performance
print("\nLSTM Metrics:")
print(f"Mean Accuracy: {np.mean(cv_results['lstm_val_accuracy']):.4f} (±{np.std(cv_results['lstm_val_accuracy']):.4f})")
print(f"Mean Balanced Accuracy: {np.mean(cv_results['lstm_val_balanced_accuracy']):.4f} (±{np.std(cv_results['lstm_val_balanced_accuracy']):.4f})")

# RF Performance
print("\nRF Metrics:")
print(f"Mean Accuracy: {np.mean(cv_results['rf_val_accuracy']):.4f} (±{np.std(cv_results['rf_val_accuracy']):.4f})")
print(f"Mean Balanced Accuracy: {np.mean(cv_results['rf_val_balanced_accuracy']):.4f} (±{np.std(cv_results['rf_val_balanced_accuracy']):.4f})")

# Ensemble Performance
print("\nEnsemble Metrics:")
print(f"Mean Accuracy: {np.mean(cv_results['ensemble_val_accuracy']):.4f} (±{np.std(cv_results['ensemble_val_accuracy']):.4f})")
print(f"Mean Balanced Accuracy: {np.mean(cv_results['ensemble_val_balanced_accuracy']):.4f} (±{np.std(cv_results['ensemble_val_balanced_accuracy']):.4f})")

# Per-class metrics
print("\nLSTM Per-Class Metrics (Averaged):")
avg_precision = np.mean(cv_results['lstm_val_precision_per_class'], axis=0)
avg_recall = np.mean(cv_results['lstm_val_recall_per_class'], axis=0)
avg_f1 = np.mean(cv_results['lstm_val_f1_per_class'], axis=0)
for i, (prec, rec, f1) in enumerate(zip(avg_precision, avg_recall, avg_f1)):
    print(f"Class {label_encoder.classes_[i]} - Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

print("\nRF Per-Class Metrics (Averaged):")
avg_precision = np.mean(cv_results['rf_val_precision_per_class'], axis=0)
avg_recall = np.mean(cv_results['rf_val_recall_per_class'], axis=0)
avg_f1 = np.mean(cv_results['rf_val_f1_per_class'], axis=0)
for i, (prec, rec, f1) in enumerate(zip(avg_precision, avg_recall, avg_f1)):
    print(f"Class {label_encoder.classes_[i]} - Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

print("\nEnsemble Per-Class Metrics (Averaged):")
avg_precision = np.mean([f1 for f1 in cv_results['ensemble_val_f1_per_class'] if len(f1) > 0], axis=0)
for i, f1 in enumerate(avg_precision):
    print(f"Class {label_encoder.classes_[i]} - F1: {f1:.4f}")

# Feature Importance
print("\nTop 10 RF Features (from last fold):")
importances = pd.DataFrame({
    'Feature': features,
    'Importance': cv_results['rf_models'][-1].named_steps['rf'].feature_importances_
}).sort_values('Importance', ascending=False)
print(importances.head(10))


=== Fold 1/5 ===


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/masking.py:47: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - accuracy: 0.3209 - loss: 6.0992 - precision: 0.5406 - recall: 0.1395

25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 380ms/step - accuracy: 0.3256 - loss: 6.0850 - precision: 0.5472 - recall: 0.1423 - val_accuracy: 0.9200 - val_loss: 5.3054 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step - accuracy: 0.7166 - loss: 4.7289 - precision: 0.8853 - recall: 0.4288

25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 276ms/step - accuracy: 0.7181 - loss: 4.7192 - precision: 0.8869 - recall: 0.4317 - val_accuracy: 0.9350 - val_loss: 4.2884 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step - accuracy: 0.8480 - loss: 3.7514 - precision: 0.9357 - recall: 0.6666

...

25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 276ms/step - accuracy: 0.8861 - loss: 3.7742 - precision: 0.9738 - recall: 0.6261 - val_accuracy: 0.8700 - val_loss: 3.5815 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 283ms/step - accuracy: 0.9432 - loss: 3.0765 - precision: 0.9673 - recall: 0.7619 - val_accuracy: 0.8550 - val_loss: 3.0116 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 5/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step - accuracy: 0.9772 - loss: 2.5554 - precision: 0.9930 - recall: 0.8065

25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 327ms/step - accuracy: 0.9771 - loss: 2.5515 - precision: 0.9930 - recall: 0.8074 - val_accuracy: 0.9050 - val_loss: 2.5630 - val_precision: 1.0000 - val_recall: 0.0050 - learning_rate: 0.0010
Epoch 6/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - accuracy: 0.9817 - loss: 2.1546 - precision: 0.9951 - recall: 0.8416

25/25 ━━━━━━━━━━━━━━━━━━━━ 10s 329ms/step - accuracy: 0.9816 - loss: 2.1518 - precision: 0.9950 - recall: 0.8419 - val_accuracy: 0.9850 - val_loss: 2.1822 - val_precision: 1.0000 - val_recall: 0.0250 - learning_rate: 0.0010
Epoch 7/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 277ms/step - accuracy: 0.9852 - loss: 1.8123 - precision: 0.9992 - recall: 0.8872 - val_accuracy: 0.9700 - val_loss: 1.8863 - val_precision: 1.0000 - val_recall: 0.0500 - learning_rate: 0.0010
Epoch 8/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step - accuracy: 0.9882 - loss: 1.5529 - precision: 0.9992 - recall: 0.8871

25/25 ━━━━━━━━━━━━━━━━━━━━ 10s 276ms/step - accuracy: 0.9881 - loss: 1.5508 - precision: 0.9992 - recall: 0.8874 - val_accuracy: 0.9900 - val_loss: 1.6146 - val_precision: 1.0000 - val_recall: 0.0950 - learning_rate: 0.0010
Epoch 9/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.9892 - loss: 1.3304 - precision: 1.0000 - recall: 0.9298

25/25 ━━━━━━━━━━━━━━━━━━━━ 10s 267ms/step - accuracy: 0.9892 - loss: 1.3287 - precision: 1.0000 - recall: 0.9298 - val_accuracy: 1.0000 - val_loss: 1.3894 - val_precision: 1.0000 - val_recall: 0.1200 - learning_rate: 0.0010
Epoch 10/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 290ms/step - accuracy: 0.9964 - loss: 1.1456 - precision: 0.9987 - recall: 0.9455 - val_accuracy: 1.0000 - val_loss: 1.1990 - val_precision: 1.0000 - val_recall: 0.1600 - learning_rate: 0.0010
Epoch 11/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 319ms/step - accuracy: 0.9988 - loss: 0.9917 - precision: 0.9987 - recall: 0.9333 - val_accuracy: 1.0000 - val_loss: 1.0399 - val_precision: 1.0000 - val_recall: 0.2100 - learning_rate: 0.0010
Epoch 12/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 279ms/step - accuracy: 0.9982 - loss: 0.8602 - precision: 0.9987 - recall: 0.9513 - val_accuracy: 0.9950 - val_loss: 0.9000 - val_precision: 1.0000 - val_recall: 0.2250 - learning_rate: 0.0010
Epoch 13/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 10s 271ms/step - accuracy: